# 03 · Evaluate runs

For each run: the decision threshold is chosen on **validation** (highest recall with FPR ≤ 0.2) and applied unchanged to **test**.
Results go to `DRIVE_ROOT/runs/<run>/eval/` (`summary.json`, `per_method_test.csv`, per-video predictions).

In [ ]:
#@title 0 · Setup (run after every restart)
DRIVE_ROOT = "/content/drive/MyDrive/IS665_Deepfake"  #@param {type:"string"}
REPO_URL = "https://github.com/payelsarkarr/is665-team6-deepfake-recognition"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
REPO = "/content/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)       # newest code from GitHub
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, f"{REPO}/src")
import cv2
if not hasattr(cv2, "CascadeClassifier"):
    !pip -q install "opencv-python-headless<5"
    print("OpenCV 4 installed -> Runtime > Restart session, then run this cell again.")
META = "/content/FaceForensics/dataset"            # official split files
if not os.path.exists(META):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ondyari/FaceForensics.git", "/content/FaceForensics"], check=True)
import importlib, ffpp
for m in ["data", "crops", "dataset", "models", "metrics", "train", "evaluate", "download"]:
    importlib.reload(importlib.import_module(f"ffpp.{m}"))
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 1 · Settings
CACHE_VERSION = "crops_v1"  #@param {type:"string"}
#@markdown Comma-separated run names (empty = all runs in DRIVE_ROOT/runs):
RUNS = ""  #@param {type:"string"}
MAX_FPR = 0.2  #@param {type:"number"}
RUNS_DIR = f"{DRIVE_ROOT}/runs"
from pathlib import Path
names = [r.strip() for r in RUNS.split(",") if r.strip()] or sorted(p.parent.name for p in Path(RUNS_DIR).glob("*/best.pt"))
print("runs:", names)

In [ ]:
#@title 2 · Copy cache to the runtime
from ffpp.dataset import copy_cache_local
LOCAL_CACHE = copy_cache_local(f"{DRIVE_ROOT}/cache/{CACHE_VERSION}", f"/content/cache/{CACHE_VERSION}")

In [ ]:
#@title 3 · Evaluate
import json, pandas as pd
from ffpp.evaluate import evaluate_run, compare_runs
for n in names:
    r = evaluate_run(f"{RUNS_DIR}/{n}", LOCAL_CACHE, max_fpr=MAX_FPR)
    t = r["test"]["at_val_threshold"]
    print(f"{n:15s} epoch {r['epoch']:2d} | test ROC-AUC {r['test']['at_0_5']['roc_auc']:.3f} | "
          f"threshold {r['threshold_from_val']:.3f} -> recall {t['recall']:.3f}, FPR {t['fpr']:.3f}")
compare_runs(RUNS_DIR)

In [ ]:
#@title 4 · Per method (test): share flagged as fake (= recall; for "original" = FPR) and ROC-AUC vs. originals
for n in names:
    print(n); display(pd.read_csv(f"{RUNS_DIR}/{n}/eval/per_method_test.csv"))

In [ ]:
#@title 5 · Leave-one-method-out: seen vs. unseen manipulation (test)
rows = []
for n in names:
    r = json.load(open(f"{RUNS_DIR}/{n}/eval/summary.json"))
    if not r.get("heldout_methods"):
        continue
    s, h = r["test_seen"], r["test_heldout"]
    rows.append(dict(run=n, heldout=",".join(r["heldout_methods"]),
                     seen_auc=s["at_0_5"]["roc_auc"], heldout_auc=h["at_0_5"]["roc_auc"],
                     seen_recall=s["at_val_threshold"]["recall"], heldout_recall=h["at_val_threshold"]["recall"],
                     fpr=s["at_val_threshold"]["fpr"]))
display(pd.DataFrame(rows).round(3) if rows else "no LOMO runs evaluated yet")

In [ ]:
#@title 6 · Confusion matrices (test, threshold from validation)
for n in names:
    c = json.load(open(f"{RUNS_DIR}/{n}/eval/summary.json"))["test"]["at_val_threshold"]
    print(f"{n}\n               predicted real  predicted fake\n  real (orig)  {c['tn']:>14d}  {c['fp']:>14d}"
          f"\n  fake         {c['fn']:>14d}  {c['tp']:>14d}\n  FPR {c['fpr']:.3f}  FNR {1 - c['recall']:.3f}\n")